<a href="https://colab.research.google.com/github/kevengoncalves-git/tecnicas_adversarias/blob/Keven/Explora%C3%A7%C3%A3o_Code_Caves.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Código Geral

## Bibliotecas utilizadas

In [1]:
!pip install pefile
!pip install capstone
!pip install keystone-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.7 MB/s eta 0:00:00


In [18]:
import pefile, os
from capstone import *
from keystone import *
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Extração dos arquivos

In [4]:
pe_path = '/content/drive/MyDrive/arquivos_RG/Goodwares/goodware_training/goodware'
goodwares = os.listdir(pe_path)
goodwares_path = [os.path.join(pe_path, goodware) for goodware in goodwares]

In [5]:
print(goodwares[:5])
print(goodwares_path[:5])

['dstokenclean.exe', 'AppHostRegistrationVerifier.exe', 'dispdiag.exe', 'dxgiadaptercache.exe', 'MicrosoftEdgeSH.exe']
['/content/drive/MyDrive/arquivos_RG/Goodwares/goodware_training/goodware/dstokenclean.exe', '/content/drive/MyDrive/arquivos_RG/Goodwares/goodware_training/goodware/AppHostRegistrationVerifier.exe', '/content/drive/MyDrive/arquivos_RG/Goodwares/goodware_training/goodware/dispdiag.exe', '/content/drive/MyDrive/arquivos_RG/Goodwares/goodware_training/goodware/dxgiadaptercache.exe', '/content/drive/MyDrive/arquivos_RG/Goodwares/goodware_training/goodware/MicrosoftEdgeSH.exe']


In [6]:
pe = pefile.PE(goodwares_path[0])

## Uso da biblioteca lief - gemini

In [92]:
!pip install lief

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.4 MB/s eta 0:00:00


**código gemini - uso do lief**

In [96]:
import lief
import os

# Altere este caminho para o seu arquivo PE (ex: um .exe ou .dll)
PE_FILE_PATH = "/content/drive/MyDrive/arquivos_RG/Goodwares/goodware_training/goodware/dstokenclean.exe"

In [113]:

binary = lief.PE.parse(PE_FILE_PATH)

if binary is None:
  print(f"ERRO: Não foi possível analisar o arquivo PE: {PE_FILE_PATH}")
  exit()

    # Verifica se é um PE e qual sua arquitetura
print("=" * 50)
print(f"ANÁLISE DE METADADOS DO ARQUIVO: {os.path.basename(PE_FILE_PATH)}")
print("=" * 50)

    # --- 1. CABEÇALHOS GERAIS ---
print("\n[INFORMAÇÕES GERAIS]")
print(f"  Tipo de Arquivo: PE {binary.header.machine.name}")
print(f"  Image Base (VA): {hex(binary.optional_header.imagebase)}")
    # O Endereço de Entrada é crucial para saber onde a execução começa.
print(f"  Ponto de Entrada (RVA): {hex(binary.optional_header.addressof_entrypoint)}")
print(f"  É 64-bit (PE+): {binary.optional_header.magic == lief.PE.PE_TYPE.PE32_PLUS}")

    # --- 2. SEÇÕES ---
print("\n[SEÇÕES DO ARQUIVO]")
for section in binary.sections:
  # A combinação RVA + Image Base dá o VA (Virtual Address)
  section_va = binary.optional_header.imagebase + section.virtual_address

        # O .text é a seção de código, e .data/.rdata são seções de dados.
        # Virtual Size é o tamanho do código/dados na memória.
  print(f"  Nome: {section.name:<8} | RVA: {hex(section.virtual_address):<10} | Tamanho (Memória): {hex(section.virtual_size)}")


# --- 3. TABELA DE IMPORTAÇÕES (IAT) ---
  print("\n[IMPORTAÇÕES (Funções Externas)]")
  if binary.has_imports:
      for imported_library in binary.imports:
          print(f"  - DLL: {imported_library.name}")
          for func in imported_library.entries:
              if func.name:
                    # Função importada por nome
                  print(f"    -> {func.name}")
              else:
                    # Função importada por ordinal (raro, mas acontece)
                  print(f"    -> Ordinal #{func.ordinal}")
  else:
        print("  O binário não possui imports (pode ser estaticamente linkado ou packed).")

# --- 4. DATA DIRECTORIES (Tabelas Cruciais) ---
  print("\n[DIRETÓRIOS DE DADOS]")
    # Lista o RVA e o tamanho de tabelas importantes
  data_directories = {
      lief.PE.DataDirectory.TYPES.IMPORT_TABLE: "Importação (IAT)",
      lief.PE.DataDirectory.TYPES.EXPORT_TABLE: "Exportação (EAT)",
      lief.PE.DataDirectory.TYPES.RESOURCE_TABLE: "Recursos (Icones, Versão)",
      lief.PE.DataDirectory.TYPES.EXCEPTION_TABLE: "Tabela de Exceções (Importante em x64)",
      lief.PE.DataDirectory.TYPES.TLS_TABLE: "TLS (Thread Local Storage)",
    }

  for dd_type, name in data_directories.items():
      dd = binary.data_directory(dd_type)
      if dd.rva != 0:
          print(f"  {name:<25}: RVA={hex(dd.rva):<10} | Tamanho={hex(dd.size)}")

ANÁLISE DE METADADOS DO ARQUIVO: dstokenclean.exe

[INFORMAÇÕES GERAIS]
  Tipo de Arquivo: PE AMD64
  Image Base (VA): 0x140000000
  Ponto de Entrada (RVA): 0x18c0
  É 64-bit (PE+): True

[SEÇÕES DO ARQUIVO]
  Nome: .text    | RVA: 0x1000     | Tamanho (Memória): 0x106a

[IMPORTAÇÕES (Funções Externas)]
  - DLL: msvcrt.dll
    -> _XcptFilter
    -> _exit
    -> exit
    -> __set_app_type
    -> _initterm
    -> __wgetmainargs
    -> _fmode
    -> _commode
    -> _cexit
    -> __C_specific_handler
    -> _vsnwprintf
    -> ?terminate@@YAXXZ
    -> _lock
    -> _unlock
    -> _amsg_exit
    -> _onexit
    -> __setusermatherr
    -> __dllonexit
    -> memset
  - DLL: api-ms-win-core-synch-l1-2-0.dll
    -> InitOnceBeginInitialize
    -> Sleep
    -> InitOnceComplete
  - DLL: api-ms-win-core-synch-l1-1-0.dll
    -> CreateMutexW
  - DLL: api-ms-win-core-errorhandling-l1-1-0.dll
    -> UnhandledExceptionFilter
    -> SetUnhandledExceptionFilter
    -> GetLastError
  - DLL: api-ms-win-core-ha

## Funções Criadas

In [7]:
def verifica_arquitetura(pe):
  if hex(pe.OPTIONAL_HEADER.Magic) == 0x10b:
    return CS_ARCH_X86, CS_MODE_32
  else:
    return CS_ARCH_X86, CS_MODE_64

In [8]:
verifica_arquitetura(pe)

(3, 8)

## Manipulação do PE

In [9]:
pe.sections

[<Structure: [IMAGE_SECTION_HEADER] 0x1F0 0x0 Name: .text 0x1F8 0x8 Misc: 0x106A 0x1F8 0x8 Misc_PhysicalAddress: 0x106A 0x1F8 0x8 Misc_VirtualSize: 0x106A 0x1FC 0xC VirtualAddress: 0x1000 0x200 0x10 SizeOfRawData: 0x1200 0x204 0x14 PointerToRawData: 0x400 0x208 0x18 PointerToRelocations: 0x0 0x20C 0x1C PointerToLinenumbers: 0x0 0x210 0x20 NumberOfRelocations: 0x0 0x212 0x22 NumberOfLinenumbers: 0x0 0x214 0x24 Characteristics: 0x60000020>,
 <Structure: [IMAGE_SECTION_HEADER] 0x218 0x0 Name: .rdata 0x220 0x8 Misc: 0x1076 0x220 0x8 Misc_PhysicalAddress: 0x1076 0x220 0x8 Misc_VirtualSize: 0x1076 0x224 0xC VirtualAddress: 0x3000 0x228 0x10 SizeOfRawData: 0x1200 0x22C 0x14 PointerToRawData: 0x1600 0x230 0x18 PointerToRelocations: 0x0 0x234 0x1C PointerToLinenumbers: 0x0 0x238 0x20 NumberOfRelocations: 0x0 0x23A 0x22 NumberOfLinenumbers: 0x0 0x23C 0x24 Characteristics: 0x40000040>,
 <Structure: [IMAGE_SECTION_HEADER] 0x240 0x0 Name: .data 0x248 0x8 Misc: 0x680 0x248 0x8 Misc_PhysicalAddress: 

In [85]:
# definição da seção que será modificada
for section in pe.sections:
  if section.Name.decode().startswith('.text'):
    text_section = section
    raw_code = text_section.get_data()
    base_rva = text_section.VirtualAddress

In [86]:
print(hex(base_rva))
#offset = base_rva+pe.OPTIONAL_HEADER.ImageBase

0x1000


In [87]:
raw_code

b'\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xccH\x8d\r\xf9\x0f\x00\x00\xe9\xa0\x0c\x00\x00\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xccH\x89\\$\x08H\x89t$\x10WH\x83\xec0H\x8b\xf2L\x8d\x05C$\x00\x003\xd23\xc93\xdb\xff\x15g!\x00\x00H\x85\xc0\x8b\xfbH\x0fE\xf8\xff\x15\xd0 \x00\x00=\xb7\x00\x00\x00u,\xe8\xa8\x00\x00\x00H\x8b\xc8L\x8d\r\xe2"\x00\x00H\x8b\x06D\x8dC&H\x8d\x15l#\x00\x00H\x89D$ \xe8\xca\x03\x00\x00\x8ds\x01\xeb\x0c\xff\x15\x87!\x00\x00\x8b\xf0\x85\xc0y\'\xe8p\x00\x00\x00H\x8b\xc8\x89t$ L\x8d\r\xbe#\x00\x00A\xb84\x00\x00\x00H\x8d\x151#\x00\x00\xe8\x94\x03\x00\x00\xeb!\xe8I\x00\x00\x00H\x8b\xc8L\x8d\r3#\x00\x00A\xb8.\x00\x00\x00H\x8d\x15\x0e#\x00\x00\xe8)\x04\x00\x00\x85\xf6\x0fI\xf3H\x85\xfft\tH\x8b\xcf\xff\x15F \x00\x00H\x8b\\$@\x8b\xc6H\x8bt$HH\x83\xc40_\xc3\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xccH\x8b\xc4H\x89H\x08SH\x83\xec H\x83`\x10\x00L\x8dH\x10L\x8d@\x083\xd2H\x8d\r\x1aE\x00\x00\xff\x15\x8c \x00\x00H\x8b\\$8H\x85\xdbu:!\x1d\x0cE\x00\x00\xe8\xb3\x01\x00\x00\x8b\x15\x01E\x00

## Disassembler com o Capstone

In [122]:
arquitetura, modo = verifica_arquitetura(pe)

In [124]:
#primeiro argumento é o tipo de arquitetura x86 e o segundo é o modo 64-bit -> visto no pe.OPTIONALHEADER.Magic
disassembler = Cs(arquitetura, modo)
disassembler.skipdata = True
#Bytes correspondentes: {str(bytes(instr.bytes)):^10}

for instr in disassembler.disasm(raw_code, base_rva):
  print(f"0x{instr.address:x}:{instr.mnemonic}\t{instr.op_str}{' '*30}")

0x1000:int3	                              
0x1001:int3	                              
0x1002:int3	                              
0x1003:int3	                              
0x1004:int3	                              
0x1005:int3	                              
0x1006:int3	                              
0x1007:int3	                              
0x1008:int3	                              
0x1009:int3	                              
0x100a:int3	                              
0x100b:int3	                              
0x100c:int3	                              
0x100d:int3	                              
0x100e:int3	                              
0x100f:int3	                              
0x1010:lea	rcx, [rip + 0xff9]                              
0x1017:jmp	0x1cbc                              
0x101c:int3	                              
0x101d:int3	                              
0x101e:int3	                              
0x101f:int3	                              
0x1020:int3	                    

In [125]:
df_disassembler = pd.DataFrame(columns=['address', 'mnemonic', 'op_str', 'bytes_correspondentes'])

for instr in disassembler.disasm(raw_code, base_rva):
  nova_linha = {'address': hex(instr.address), 'mnemonic': instr.mnemonic, 'op_str': instr.op_str, 'bytes_correspondentes': bytes(instr.bytes)}
  df_disassembler = pd.concat([df_disassembler, pd.DataFrame([nova_linha])], ignore_index=True)

In [78]:
df_disassembler.head(20)

,address,mnemonic,op_str,bytes_correspondentes
0,0x1000,int3,,b'\xcc'
1,0x1001,int3,,b'\xcc'
2,0x1002,int3,,b'\xcc'
3,0x1003,int3,,b'\xcc'
4,0x1004,int3,,b'\xcc'
5,0x1005,int3,,b'\xcc'
6,0x1006,int3,,b'\xcc'
7,0x1007,int3,,b'\xcc'
8,0x1008,int3,,b'\xcc'
9,0x1009,int3,,b'\xcc'


## Remontagem com a biblioteca Keystone

**Montagem e demonstagem sem modificações**

In [126]:
#dados básicos

#raw_code -> dados obtidos de dentro da seção escolhida

print(raw_code)
print(hex(base_rva))

assembly_code = '; '.join([f"{instr.mnemonic} {instr.op_str}" for instr in disassembler.disasm(raw_code, base_rva)])
print(assembly_code)


b'\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xccH\x8d\r\xf9\x0f\x00\x00\xe9\xa0\x0c\x00\x00\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xccH\x89\\$\x08H\x89t$\x10WH\x83\xec0H\x8b\xf2L\x8d\x05C$\x00\x003\xd23\xc93\xdb\xff\x15g!\x00\x00H\x85\xc0\x8b\xfbH\x0fE\xf8\xff\x15\xd0 \x00\x00=\xb7\x00\x00\x00u,\xe8\xa8\x00\x00\x00H\x8b\xc8L\x8d\r\xe2"\x00\x00H\x8b\x06D\x8dC&H\x8d\x15l#\x00\x00H\x89D$ \xe8\xca\x03\x00\x00\x8ds\x01\xeb\x0c\xff\x15\x87!\x00\x00\x8b\xf0\x85\xc0y\'\xe8p\x00\x00\x00H\x8b\xc8\x89t$ L\x8d\r\xbe#\x00\x00A\xb84\x00\x00\x00H\x8d\x151#\x00\x00\xe8\x94\x03\x00\x00\xeb!\xe8I\x00\x00\x00H\x8b\xc8L\x8d\r3#\x00\x00A\xb8.\x00\x00\x00H\x8d\x15\x0e#\x00\x00\xe8)\x04\x00\x00\x85\xf6\x0fI\xf3H\x85\xfft\tH\x8b\xcf\xff\x15F \x00\x00H\x8b\\$@\x8b\xc6H\x8bt$HH\x83\xc40_\xc3\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xccH\x8b\xc4H\x89H\x08SH\x83\xec H\x83`\x10\x00L\x8dH\x10L\x8d@\x083\xd2H\x8d\r\x1aE\x00\x00\xff\x15\x8c \x00\x00H\x8b\\$8H\x85\xdbu:!\x1d\x0cE\x00\x00\xe8\xb3\x01\x00\x00\x8b\x15\x01E\x00

In [80]:
assembly_remontado_bytes = bytes(assembly_code, 'utf-8')
print(assembly_remontado_bytes)

b'int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; lea rcx, [rip + 0xff9]; jmp 0x1cbc; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; int3 ; mov qword ptr [rsp + 8], rbx; mov qword ptr [rsp + 0x10], rsi; push rdi; sub rsp, 0x30; mov rsi, rdx; lea r8, [rip + 0x2443]; xor edx, edx; xor ecx, ecx; xor ebx, ebx; call qword ptr [rip + 0x2167]; test rax, rax; mov edi, ebx; cmovne rdi, rax; call qword ptr [rip + 0x20d0]; cmp eax, 0xb7; jne 0x108b; call 0x110c; mov rcx, rax; lea r9, [rip + 0x22e2]; mov rax, qword ptr [rsi]; lea r8d, [rbx + 0x26]; lea rdx, [rip + 0x236c]; mov qword ptr [rsp + 0x20], rax; call 0x1450; lea esi, [rbx + 1]; jmp 0x1097; call qword ptr [rip + 0x2187]; mov esi, eax; test eax, eax; jns 0x10be; call 0x110c; mov rcx, rax; mov dword ptr [rsp + 0x20], esi; lea r9, [rip + 0x23be]; mov r8d, 0x34; lea rdx, [rip + 0x2331]; call 0x1450; jmp 0x10df; call 0x110c; mov rcx, rax; lea r9, [rip + 0x2333]; mov r8d, 0x2e; l

In [81]:
assembler = Ks(KS_ARCH_X86, KS_MODE_64)
assembler.syntax = KS_OPT_SYNTAX_INTEL

try:
  encoding, count = assembler.asm(assembly_remontado_bytes, base_rva)
  bytes_remontados = bytes(encoding)
  print("\n--- RESULTADO FINAL ---")
  print(f"Total de instruções remontadas: {count}")
  print(f"Bytes Remontados (Hex): {bytes_remontados}")

except Exception as e:
    print(f"\n[ERRO] Falha na montagem com Keystone: {e}")


--- RESULTADO FINAL ---
Total de instruções remontadas: 1602
Bytes Remontados (Hex): b'\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xccH\x8d\r\xf9\x0f\x00\x00\xe9\xa0\x0c\x00\x00\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xccH\x89\\$\x08H\x89t$\x10WH\x83\xec0H\x89\xd6L\x8d\x05C$\x00\x001\xd21\xc91\xdb\xff\x15g!\x00\x00H\x85\xc0\x89\xdfH\x0fE\xf8\xff\x15\xd0 \x00\x00=\xb7\x00\x00\x00u,\xe8\xa8\x00\x00\x00H\x89\xc1L\x8d\r\xe2"\x00\x00H\x8b\x06D\x8dC&H\x8d\x15l#\x00\x00H\x89D$ \xe8\xca\x03\x00\x00\x8ds\x01\xeb\x0c\xff\x15\x87!\x00\x00\x89\xc6\x85\xc0y\'\xe8p\x00\x00\x00H\x89\xc1\x89t$ L\x8d\r\xbe#\x00\x00A\xb84\x00\x00\x00H\x8d\x151#\x00\x00\xe8\x94\x03\x00\x00\xeb!\xe8I\x00\x00\x00H\x89\xc1L\x8d\r3#\x00\x00A\xb8.\x00\x00\x00H\x8d\x15\x0e#\x00\x00\xe8)\x04\x00\x00\x85\xf6\x0fI\xf3H\x85\xfft\tH\x89\xf9\xff\x15F \x00\x00H\x8b\\$@\x89\xf0H\x8bt$HH\x83\xc40_\xc3\xcc\xcc\xcc\xcc\xcc\xcc\xcc\xccH\x89\xe0H\x89H\x08SH\x83\xec H\x83`\x10\x00L\x8dH\x10L\x8d@\x081\xd2H\x8d\r\x1aE\x00\x00\xff\x15\x

In [82]:
raw_code == bytes_remontados

False

In [83]:
len(bytes_remontados)

4589

In [84]:
len(raw_code)

4608

In [127]:
for instr in disassembler.disasm(bytes_remontados, base_rva):
  print(f'[{hex(instr.address)}]: {instr.mnemonic} \t {instr.op_str}')

[0x1000]: int3 	 
[0x1001]: int3 	 
[0x1002]: int3 	 
[0x1003]: int3 	 
[0x1004]: int3 	 
[0x1005]: int3 	 
[0x1006]: int3 	 
[0x1007]: int3 	 
[0x1008]: int3 	 
[0x1009]: int3 	 
[0x100a]: int3 	 
[0x100b]: int3 	 
[0x100c]: int3 	 
[0x100d]: int3 	 
[0x100e]: int3 	 
[0x100f]: int3 	 
[0x1010]: lea 	 rcx, [rip + 0xff9]
[0x1017]: jmp 	 0x1cbc
[0x101c]: int3 	 
[0x101d]: int3 	 
[0x101e]: int3 	 
[0x101f]: int3 	 
[0x1020]: int3 	 
[0x1021]: int3 	 
[0x1022]: int3 	 
[0x1023]: int3 	 
[0x1024]: mov 	 qword ptr [rsp + 8], rbx
[0x1029]: mov 	 qword ptr [rsp + 0x10], rsi
[0x102e]: push 	 rdi
[0x102f]: sub 	 rsp, 0x30
[0x1033]: mov 	 rsi, rdx
[0x1036]: lea 	 r8, [rip + 0x2443]
[0x103d]: xor 	 edx, edx
[0x103f]: xor 	 ecx, ecx
[0x1041]: xor 	 ebx, ebx
[0x1043]: call 	 qword ptr [rip + 0x2167]
[0x1049]: test 	 rax, rax
[0x104c]: mov 	 edi, ebx
[0x104e]: cmovne 	 rdi, rax
[0x1052]: call 	 qword ptr [rip + 0x20d0]
[0x1058]: cmp 	 eax, 0xb7
[0x105d]: jne 	 0x108b
[0x105f]: call 	 0x110c
[0x1064

In [129]:
for instr in disassembler.disasm(bytes_remontados, base_rva):
  nova_linha = {'address': hex(instr.address), 'mnemonic': instr.mnemonic, 'op_str': instr.op_str, 'bytes_correspondentes': bytes(instr.bytes)}
  df_assembler = pd.concat([df_disassembler, pd.DataFrame([nova_linha])], ignore_index=True)

In [130]:
df_assembler

,address,mnemonic,op_str,bytes_correspondentes
0,0x1000,int3,,b'\xcc'
1,0x1001,int3,,b'\xcc'
2,0x1002,int3,,b'\xcc'
3,0x1003,int3,,b'\xcc'
4,0x1004,int3,,b'\xcc'
...,...,...,...,...
1597,0x21f8,add,"byte ptr [rax], al",b'\x00\x00'
1598,0x21fa,add,"byte ptr [rax], al",b'\x00\x00'
1599,0x21fc,add,"byte ptr [rax], al",b'\x00\x00'
1600,0x21fe,add,"byte ptr [rax], al",b'\x00\x00'
